# Urban green accessibility in Turin
## Do residents of Turin have equal access to green spaces?

This notebook analyses the spatial relationship between residential buildings 
and publicly accessible green spaces across Turin's 8 administrative districts (*circoscrizioni*).

**Data sources**
- Overture Maps (2026-02-18) — residential buildings
- OpenStreetMap via Overpass Turbo — parks, gardens, forests, recreation areas
- Comune di Torino — administrative boundaries (EPSG:3003)

**Tools**
- DuckDB + spatial extension — spatial queries and ST_DWithin analysis
- QGIS — green layer extraction and clipping
- Python / Pandas — data wrangling and export

**Methodology notes**
- Green access defined as at least one green space within 300m walking distance
- Analysis does not distinguish between public and private green spaces
- Private gardens and enclosed parks may inflate accessibility figures in hillside districts
- AI assistance: Claude (Anthropic)


In [ ]:
BASE_DIR = 'C:/Users/Sonia/Desktop/torino_green/'
SHP_DIR = 'C:/Users/Sonia/Desktop/circoscrizioni_geo/'

In [ ]:
import duckdb
import os

In [ ]:
con = duckdb.connect()
con.execute("INSTALL spatial; LOAD spatial;")
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute("SET s3_region='us-west-2';")

In [ ]:
con.execute ("""
SELECT *
FROM duckdb_extensions ()
WHERE loaded=true;
""").df()

In [ ]:
con.execute ("SET s3_access_key_id='';")

In [ ]:
con.execute("SET s3_secret_access_key ='';")

In [ ]:
df_building_torino = con.execute ("""
SELECT id, subtype, class,
ST_AsText (geometry) AS geom_wkt
FROM read_parquet ('s3://overturemaps-us-west-2/release/2026-02-18.0/theme=buildings/type=building/*',hive_partitioning=false)
WHERE subtype='residential'
AND class NOT IN ('garage', 'garages', 'parking')
AND bbox.xmin BETWEEN 7.6 AND 7.72
AND bbox.ymin BETWEEN 44.99 AND 45.12
""").df()

In [ ]:
os.makedirs('{BASE_DIR}', exist_ok=True)

In [ ]:
os.makedirs('{BASE_DIR}', exist_ok=True)
print(os.path.exists('{BASE_DIR}'))

In [ ]:
df_building_torino.to_parquet('{BASE_DIR}buildings_raw.parquet')

In [ ]:
print("salvato")

In [ ]:
print(df_building_torino.shape)
print(df_building_torino.columns.tolist())
print(df_building_torino.head())

In [ ]:
torino_boundary = con.execute (f"""
SELECT ST_AsText (
ST_Transform (
ST_Union_Agg(geom),
'EPSG:3003',
'EPSG:4326',
true
)
) AS geom_wkt
FROM ST_ReadSHP ('{SHP_DIR}circoscrizioni_geo.shp', encoding='UTF-8')
""").df()


In [ ]:
print(torino_boundary['geom_wkt'].iloc[0][:100])

In [ ]:
con.register ('buildings', df_building_torino)

In [ ]:
con.register ('boundary', torino_boundary)

In [ ]:
df_buildings_filtered = con.execute("""
    SELECT b.id, b.subtype, b.class, b.geom_wkt
    FROM buildings b, boundary bo
    WHERE ST_Within(
        ST_GeomFromText(b.geom_wkt),
        ST_GeomFromText(bo.geom_wkt)
    )
""").df()

print(len(df_buildings_filtered))

In [ ]:
import pandas as pd
df_buildings_filtered = pd.read_parquet(f'{BASE_DIR}buildings_filtered.parquet')
torino_boundary = pd.read_csv(f'{BASE_DIR}boundary.csv')
print(len(df_buildings_filtered))
print(torino_boundary['geom_wkt'].iloc[0][:100])

In [ ]:
df_verde = con.execute(f"""
    SELECT ST_AsText(geom) AS geom_wkt
    FROM ST_Read('{BASE_DIR}verde-torino.geojson')
""").df()

In [ ]:
con.register('buildings', df_buildings_filtered)
con.register('verde', df_verde)

df_access = con.execute("""
    SELECT 
        b.id,
        b.subtype,
        b.class,
        b.geom_wkt,
        CASE WHEN COUNT(v.geom_wkt) > 0 THEN true ELSE false END AS has_green_access
    FROM buildings b
    LEFT JOIN verde v
        ON ST_DWithin(
            ST_Transform(ST_GeomFromText(b.geom_wkt), 'EPSG:4326', 'EPSG:32632', true),
            ST_Transform(ST_GeomFromText(v.geom_wkt), 'EPSG:4326', 'EPSG:32632', true),
            300
        )
    GROUP BY b.id, b.subtype, b.class, b.geom_wkt
""").df()

print(len(df_access))
print(df_access['has_green_access'].value_counts())

In [ ]:
df_circoscrizioni = con.execute(f"""
    SELECT 
        NCIRCO,
        DENOM,
        ST_AsText(
            ST_Transform(geom, 'EPSG:3003', 'EPSG:4326', true)
        ) AS geom_wkt
    FROM ST_ReadSHP('{SHP_DIR}circoscrizioni_geo.shp', encoding='UTF-8')
""").df()

print(len(df_circoscrizioni))
print(df_circoscrizioni[['NCIRCO', 'DENOM']])

In [ ]:
con.register('circoscrizioni', df_circoscrizioni)
con.register('access', df_access)

df_result = con.execute("""
    SELECT 
        c.NCIRCO,
        c.DENOM,
        COUNT(a.id) AS total_buildings,
        SUM(CASE WHEN a.has_green_access THEN 1 ELSE 0 END) AS buildings_with_green,
        ROUND(100.0 * SUM(CASE WHEN a.has_green_access THEN 1 ELSE 0 END) / COUNT(a.id), 1) AS pct_green_access
    FROM access a
    JOIN circoscrizioni c
        ON ST_Within(
            ST_GeomFromText(a.geom_wkt),
            ST_GeomFromText(c.geom_wkt)
        )
    GROUP BY c.NCIRCO, c.DENOM
    ORDER BY pct_green_access ASC
""").df()

print(df_result)

In [ ]:
df_access.to_parquet('{BASE_DIR}access.parquet')
df_result.to_parquet('{BASE_DIR}result_by_circoscrizione.parquet')
print("salvati")

In [ ]:
con.register('result_by_circoscrizione', df_result)

df_result_geo = con.execute("""
    SELECT 
        r.NCIRCO,
        r.DENOM,
        r.total_buildings,
        r.buildings_with_green,
        r.pct_green_access,
        c.geom_wkt
    FROM result_by_circoscrizione r
    JOIN circoscrizioni c ON r.NCIRCO = c.NCIRCO
""").df()

print(df_result_geo.shape)

In [ ]:
df_result_geo = con.execute("""
    SELECT 
        r.NCIRCO,
        r.DENOM,
        r.total_buildings,
        r.buildings_with_green,
        r.pct_green_access,
        c.geom_wkt
    FROM result_by_circoscrizione r
    JOIN circoscrizioni c ON r.NCIRCO = c.NCIRCO
""").df()

In [ ]:
import geopandas as gpd
from shapely import wkt

gdf = gpd.GeoDataFrame(
    df_result_geo,
    geometry=df_result_geo['geom_wkt'].apply(wkt.loads),
    crs='EPSG:4326'
)
gdf.to_file('f{BASE_DIR}result_circoscrizioni.geojson', driver='GeoJSON')
print("salvato")